# Aligner le service client par DPO

Le fil rouge de la formation : le service client de Biclou & Cie, dont le modèle doit trier chaque message (catégorie, priorité) et y répondre selon la charte, en un objet JSON. Le TP d'entraînement a produit un adaptateur QLoRA de Qwen3-1.7B, l'atelier d'évaluation l'a mesuré. Ce TP l'aligne sur une préférence métier par DPO (*Direct Preference Optimization*) : constituer un petit jeu de paires préférée / rejetée, le contrôler, entraîner, puis comparer avant et après avec le protocole de l'atelier.

Le critère métier : la conformité à la charte, ton compris (vouvoiement, salutation, signature, longueur, numéro de commande rappelé, aucune promesse de remboursement avant contrôle). Il se vérifie en grande partie par des règles, ce qui permet de constituer des paires sans annotateur, et c'est là que pèchent les réponses de Jordan, le conseiller que les clients notent le plus mal.

1. Le critère en code
2. Les candidats et les paires
3. Contrôler le jeu de préférences
4. Un juge en contrôle
5. L'alignement DPO avec `trl`
6. Avant / après : les mesures
7. Avant / après : le juge par paires

L'adaptateur de départ est le vôtre si vous l'avez publié à la fin du TP d'entraînement (le secret Colab `HF_TOKEN`), sinon un adaptateur de référence, entraîné par le même code.

Ce notebook demande un GPU : *Exécution > Modifier le type d'exécution > GPU T4*, sur lequel il tient en une heure environ (une estimation), dont un quart d'heure pour échantillonner les réponses candidates. Sans GPU, il tourne quand même, en plus petit : Qwen3-0.6B et son adaptateur au lieu de Qwen3-1.7B, 5 messages au lieu de 140, donc quelques paires, 4 tickets de test au lieu de 150 (2 pour le juge), et Qwen3-1.7B pour juge au lieu de Qwen3-4B-Instruct-2507 : 25 minutes environ sur deux processeurs, comme ceux de Colab. Sur si peu de paires, l'alignement ne change presque rien : c'est la méthode qui compte.

## Données et modèles

`support-sft-train.jsonl` : le jeu d'entraînement du SFT (1 190 exemples), d'où viennent les messages des paires, jamais du test. `tickets.csv` : l'export brut de l'outil de support, avec les réponses de Jordan. `test.jsonl` : les 150 tickets de test validés par l'équipe, pour la comparaison avant / après. `charte.md` : la charte de réponse du service client, que lit le juge.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/lab-results/support-sft-train.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/tickets.csv
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/test.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/charte.md
%pip install -q peft trl bitsandbytes
%pip uninstall -q -y torchao  # la version de Colab est trop ancienne pour peft

In [ ]:
import collections
import copy
import gc
import json
import os
import re
import time
import urllib.request

import pandas as pd
import torch
from datasets import Dataset
from huggingface_hub import HfApi
from peft import PeftModel
from transformers import (
  AutoModelForCausalLM,
  AutoTokenizer,
  BitsAndBytesConfig,
  DynamicCache,
)
from trl import DPOConfig, DPOTrainer

GPU = torch.cuda.is_available()
# bfloat16 à partir des GPU Ampere (A100, L4, H100) ; le T4 calcule en float16.
BF16 = GPU and torch.cuda.get_device_capability()[0] >= 8
# Sur CPU, float32 : les demi-précisions y sont très lentes.
DTYPE = (torch.bfloat16 if BF16 else torch.float16) if GPU else torch.float32
DEVICE = "cuda" if GPU else "cpu"
MODELE = "Qwen/Qwen3-1.7B" if GPU else "Qwen/Qwen3-0.6B"
JUGE = "Qwen/Qwen3-4B-Instruct-2507" if GPU else "Qwen/Qwen3-1.7B"
# Sans GPU, en plus petit : les messages du jeu du SFT et de Jordan, les paires
# que le juge contrôle, et les tickets de test qu'il compare.
N_SFT = 100 if GPU else 2
N_JORDAN = 40 if GPU else 3
N_CONTROLE = 30 if GPU else 2
N_DUEL = 50 if GPU else 2
print(MODELE, "sur GPU" if GPU else "sans GPU, en plus petit")

tokenizer = AutoTokenizer.from_pretrained(MODELE)
tokenizer.padding_side = "left"
entrainement_sft = pd.read_json("support-sft-train.jsonl", lines=True)
test = pd.read_json("test.jsonl", lines=True)
test_part = test[: None if GPU else 4]
with open("charte.md", encoding="utf-8") as f:
  charte = f.read()
tickets = pd.read_csv("tickets.csv").drop_duplicates("ticket_id").set_index("ticket_id")
print(len(entrainement_sft), "exemples du SFT ;", len(test_part), "tickets de test ici")


def libere() -> None:
  """Rend au GPU la mémoire des objets supprimés par `del`."""
  gc.collect()
  if GPU:
    torch.cuda.empty_cache()

L'adaptateur SFT, dans le dossier `ADAPTATEUR` : le vôtre, s'il est sur votre Hub et a été entraîné sur `MODELE`, sinon celui de référence, de [shuuchuu/models](https://github.com/shuuchuu/models).

In [ ]:
ADAPTATEUR = "support-sft-lora"
REFERENCE = "https://github.com/shuuchuu/models/raw/refs/heads/main/" + (
  "support-sft-lora-qwen3-1.7b" if GPU else "support-sft-lora-qwen3-0.6b"
)


def adaptateur_du_hub() -> str | None:
  """Télécharge votre adaptateur dans ADAPTATEUR, s'il existe pour MODELE."""
  jeton = os.environ.get("HF_TOKEN")
  try:
    from google.colab import userdata

    jeton = userdata.get("HF_TOKEN")
  except Exception:
    pass
  if not jeton:
    return None
  api = HfApi(token=jeton)
  depot = f"{api.whoami()['name']}/support-sft-lora"
  if not api.repo_exists(depot):
    return None
  with open(api.hf_hub_download(depot, "adapter_config.json")) as f:
    if json.load(f)["base_model_name_or_path"] != MODELE:
      return None
  api.snapshot_download(depot, local_dir=ADAPTATEUR)
  return depot


depot = adaptateur_du_hub()
if depot is None:
  os.makedirs(ADAPTATEUR, exist_ok=True)
  for fichier in ["adapter_config.json", "adapter_model.safetensors"]:
    urllib.request.urlretrieve(f"{REFERENCE}/{fichier}", f"{ADAPTATEUR}/{fichier}")
print("Adaptateur :", depot or REFERENCE)

## La tâche, sa mesure et le juge

Le code de l'atelier d'évaluation, repris tel quel : `CONSIGNES`, la consigne courte que le modèle affiné a reçue à l'entraînement ; `lit_json` ; `respecte_charte`, les quatre règles de forme de la charte ; `evalue(sorties)`, qui mesure les sorties sur `test_part` (`json_valide`, `categorie`, `priorite`, `charte`, `mots`) ; `genere_tout(modele, conversations)`, qui génère par lots, sans échantillonnage ; `conversation_seule(message)` ; et `charge_modele()`, la solution de son exercice 1, ici en deux temps : `charge_base()`, le modèle de base, quantifié en NF4 sur GPU, puis l'adaptateur `ADAPTATEUR`.

Et son juge : `juge_tout(grille, demandes)`, qui fait lire `grille` au juge en message `system`, puis chaque demande ; la comparaison par paires, `GRILLE_PAIRE` et `demande_paire(ligne, reponse_a, reponse_b)`, avec la réponse de référence d'un ticket de test ; et sa règle, dans `juge_deux_ordres(grille, demandes, inversees)` : chaque paire est jugée dans les deux ordres, et une victoire ne compte que si les deux s'accordent. Le juge lui-même, `juge`, n'est chargé (`charge_juge()`) que le temps de s'en servir : la mémoire ne tient pas les deux modèles et un entraînement.

In [ ]:
CATEGORIES = [
  "information",
  "livraison",
  "reparation",
  "retour",
  "location",
  "facturation",
  "compte",
]
PRIORITES = ["basse", "normale", "haute"]
CONSIGNES = f"""Tu es l'assistant du service client de Biclou & Cie, magasins de vélos.
Pour le message du client, réponds uniquement par un objet JSON avec trois clés :
- categorie : une valeur parmi {", ".join(CATEGORIES)} ;
- priorite : basse, normale ou haute ;
- reponse : la réponse à envoyer au client."""
SIGNATURE = "Bonne journée,\nL'équipe Biclou & Cie"
TUTOIEMENT = re.compile(r"\b(tu|te|toi|ton|ta|tes)\b|\bt['’]", re.IGNORECASE)


def lit_json(texte: str) -> dict:
  """L'objet JSON d'une réponse du modèle, ou {} s'il n'est pas valide."""
  texte = texte.strip().removeprefix("```json").removesuffix("```").strip()
  try:
    objet = json.loads(texte)
  except json.JSONDecodeError:
    return {}
  return objet if isinstance(objet, dict) else {}


def respecte_charte(reponse: str) -> bool:
  reponse = reponse.strip()
  return (
    reponse.startswith("Bonjour")
    and reponse.endswith(SIGNATURE)
    and not TUTOIEMENT.search(reponse)
    and 40 <= len(reponse.split()) <= 120
  )


def evalue(sorties: list[str]) -> pd.Series:
  objets = [lit_json(s) for s in sorties]
  reponses = [str(o.get("reponse", "")) for o in objets]
  return (
    pd.Series(
      {
        "json_valide": sum(
          {"categorie", "priorite", "reponse"} <= o.keys() for o in objets
        ),
        "categorie": sum(
          o.get("categorie") == c for o, c in zip(objets, test_part["categorie"])
        ),
        "priorite": sum(
          o.get("priorite") == p for o, p in zip(objets, test_part["priorite"])
        ),
        "charte": sum(respecte_charte(r) for r in reponses),
        "mots": sum(len(r.split()) for r in reponses),
      }
    )
    .div(len(sorties))
    .round(2)
  )


def genere_tout(
  modele, conversations: list[list[dict]], taille_lot: int = 16, max_tokens: int = 384
) -> list[str]:
  textes = []
  for debut in range(0, len(conversations), taille_lot):
    prompts = [
      tokenizer.apply_chat_template(
        c, tokenize=False, add_generation_prompt=True, enable_thinking=False
      )
      for c in conversations[debut : debut + taille_lot]
    ]
    entrees = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
      sorties = modele.generate(
        **entrees,
        max_new_tokens=max_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
      )
    textes += tokenizer.batch_decode(
      sorties[:, entrees.input_ids.shape[1] :], skip_special_tokens=True
    )
  return textes


def conversation_seule(message: str) -> list[dict]:
  return [
    {"role": "system", "content": CONSIGNES},
    {"role": "user", "content": message},
  ]


def charge_base():
  quantification = (
    BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_use_double_quant=True,
      bnb_4bit_compute_dtype=DTYPE,
    )
    if GPU
    else None
  )
  return AutoModelForCausalLM.from_pretrained(
    MODELE, dtype=DTYPE, device_map=DEVICE, quantization_config=quantification
  )


def charge_modele() -> PeftModel:
  return PeftModel.from_pretrained(charge_base(), ADAPTATEUR).eval()

In [ ]:
tokenizer_juge = AutoTokenizer.from_pretrained(JUGE)


def charge_juge():
  return AutoModelForCausalLM.from_pretrained(JUGE, dtype=DTYPE, device_map=DEVICE)


def juge_tout(grille: str, demandes: list[str], max_tokens: int = 160) -> list[str]:
  """Les réponses du juge à chaque demande, `grille` en message system.

  Le début des prompts, la grille et sa charte, est le même pour toutes les
  demandes : il n'est calculé qu'une fois, et chaque génération repart d'une copie
  de son cache, comme le fait un serveur d'inférence (le cache de préfixe).
  """
  systeme = [{"role": "system", "content": grille}]
  debut = tokenizer_juge.apply_chat_template(systeme, tokenize=False)
  cache = DynamicCache(config=juge.config)
  with torch.no_grad():
    juge(**tokenizer_juge(debut, return_tensors="pt").to(DEVICE), past_key_values=cache)
  textes = []
  for demande in demandes:
    prompt = tokenizer_juge.apply_chat_template(
      [*systeme, {"role": "user", "content": demande}],
      tokenize=False,
      add_generation_prompt=True,
      enable_thinking=False,
    )
    entree = tokenizer_juge(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
      sortie = juge.generate(
        **entree,
        past_key_values=copy.deepcopy(cache),
        max_new_tokens=max_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
      )
    texte = sortie[0, entree.input_ids.shape[1] :]
    textes.append(tokenizer_juge.decode(texte, skip_special_tokens=True))
  return textes


GRILLE_PAIRE = f"""Tu compares deux réponses du service client de Biclou & Cie, magasins
de vélos, au même message d'un client. On te donne le message, la réponse de
référence validée par l'équipe, puis les réponses A et B. Dis laquelle suit le mieux
la charte ci-dessous et la réponse de référence : des faits exacts, une prochaine
étape claire, aucune promesse de remboursement avant contrôle, un ton courtois.
Réponds uniquement par un objet JSON : {{"meilleure": "A"}} ou {{"meilleure": "B"}}.

{charte}"""


def demande_paire(ligne, reponse_a: str, reponse_b: str) -> str:
  return (
    f"Message du client :\n{ligne.message}\n\n"
    f"Réponse de référence :\n{ligne.reponse}\n\n"
    f"Réponse A :\n{reponse_a}\n\n"
    f"Réponse B :\n{reponse_b}"
  )


def juge_deux_ordres(
  grille: str, demandes: list[str], inversees: list[str]
) -> list[str | None]:
  """Le gagnant de chaque paire, jugée dans les deux ordres.

  `demandes` présente chaque paire (x, y) avec x en A et y en B, `inversees` avec y
  en A et x en B. Pour chaque paire : "premiere" si x gagne dans les deux ordres,
  "seconde" si y gagne dans les deux, None si le verdict suit la position.
  """
  textes = juge_tout(grille, demandes + inversees, max_tokens=16)
  lettres = [lit_json(t).get("meilleure") for t in textes]
  gagnants = {("A", "B"): "premiere", ("B", "A"): "seconde"}
  return [
    gagnants.get(verdicts)
    for verdicts in zip(lettres[: len(demandes)], lettres[len(demandes) :])
  ]

## 1. Le critère en code

`respecte_charte`, de l'atelier, vérifie quatre règles d'un coup, par oui ou non. Pour classer des réponses entre elles, il faut une note graduée, règle par règle. Écrivez `note_charte(message, reponse)`, qui renvoie la note de la réponse au message, la part des règles applicables qu'elle respecte (arrondie au centième), et la liste des règles enfreintes, sous ces noms :

- `vouvoiement` : aucun tutoiement (`TUTOIEMENT`) ;
- `salutation` : la première ligne est une salutation de la charte (`SALUTATION`), dans la langue du message (`Bonjour` ou `Hello`), au nom de famille qui signe le message (`nom_signe(message)`), sans nom quand il n'en a pas ;
- `longueur` : 40 à 120 mots ;
- `signature` : elle finit par `SIGNATURE`, ou par `SIGNATURE_EN` pour un message en anglais (`en_anglais`) ;
- `sans_promesse` : aucune phrase ne promet un remboursement (`PROMESSE`) sans condition (`CONDITION`) ;
- `sans_coordonnees` : aucune phrase ne demande de coordonnées bancaires ni de mot de passe (`DEMANDE`), sauf pour dire de ne pas les envoyer (`NEGATION`) ;
- `langue` : elle est dans la langue du message ;
- `commande` : quand le message donne un numéro de commande (`COMMANDE`), elle le rappelle ; sans numéro, la règle ne compte pas.

Essayez-la sur la réponse attendue d'un exemple du jeu du SFT (`reponse_exemple`), puis sur la réponse de Jordan au ticket `T-13812` (`jordan_exemple`). Que cette note ne voit-elle pas ?

**Documentation utile**

- [`re.Pattern.search`](https://docs.python.org/3/library/re.html#re.Pattern.search)
- [`re.Pattern.fullmatch`](https://docs.python.org/3/library/re.html#re.Pattern.fullmatch)

In [ ]:
SIGNATURE_EN = "Have a nice day,\nThe Biclou & Cie team"
# Une salutation de la charte, et le nom qu'elle donne : « Bonjour Madame Martin, ».
SALUTATION = re.compile(r"(Bonjour|Hello)(?: (?:Madame|Monsieur|Ms|Mr) ([^,]+))?,")
# Un nom de famille à la fin du message : « Marion Fournier », « M. Le Goff ».
SIGNE = re.compile(
  r"(?:^|[.!?] )(?:(?:Mme|Madame|M\.|Monsieur|Mrs?|Ms) |[A-ZÉ][^\W\d_]+(?:-[A-ZÉ][^\W\d_]+)? )"
  r"((?:(?:Le|La|De|Da|Du|LE|LA|DE|DA|DU) )?[A-ZÉ][\w'-]+)\Z",
  re.MULTILINE,
)
COMMANDE = re.compile(r"CMD-\d{6}")
PROMESSE = re.compile(
  r"rembours\w* (tout de suite|immédiatement|direct)"
  r"|(je|on|nous) (t'|te |vous )(ai |avons )?rembours"
  r"|vous (êtes|avez été|serez) rembours|we('ll| will| have|'ve) refund",
  re.IGNORECASE,
)
CONDITION = re.compile(r"après|dès (réception|que)|une fois|after|once", re.IGNORECASE)
DEMANDE = re.compile(
  r"(envoy|communiqu|transmet|indiqu|donn|send|provide)\w*[^.?!]*"
  r"(RIB|IBAN|coordonnées bancaires|numéro de (votre )?carte|mot de passe|password)",
  re.IGNORECASE,
)
NEGATION = re.compile(r"inutile|jamais|\bne\b|\bn'|never|n't|do not", re.IGNORECASE)
ANGLAIS = re.compile(r"\b(the|and|my|you|is|it|to|of|hello|please|thanks?)\b", re.I)
FRANCAIS = re.compile(r"\b(le|la|les|et|mon|ma|vous|est|de|bonjour|merci)\b", re.I)


def en_anglais(texte: str) -> bool:
  """Vrai si le texte a plus de mots anglais courants que de mots français."""
  return len(ANGLAIS.findall(texte)) > len(FRANCAIS.findall(texte))


def nom_signe(message: str) -> str | None:
  """Le nom de famille qui signe le message, ou None."""
  signe = SIGNE.search(message.strip())
  return signe.group(1) if signe else None


def phrases(texte: str) -> list[str]:
  return re.split(r"(?<=[.!?])\s+", texte)


exemple = entrainement_sft.iloc[0]
message_exemple = exemple["prompt"][-1]["content"]
reponse_exemple = lit_json(exemple["completion"][0]["content"])["reponse"]
jordan_exemple = tickets.loc["T-13812"]
print(message_exemple, "\n\n", reponse_exemple)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `SALUTATION.fullmatch(premiere_ligne)` : `None` si la ligne n'est pas une salutation de la charte ; sinon `.group(1)` vaut `Bonjour` ou `Hello`, et `.group(2)` le nom, ou `None`
- Les noms se comparent sans la casse : `DAVID` signe un message, `Bonjour Madame David,` y répond ; `(salut.group(2) or "").lower() == (nom_signe(message) or "").lower()` vaut vrai aussi quand aucun nom n'est attendu ni donné
- Une règle sur les phrases : `not any(PROMESSE.search(p) and not CONDITION.search(p) for p in phrases(reponse))`
- Les règles dans un dictionnaire, `regles = dict(vouvoiement=..., ...)`, puis `regles["commande"] = ...` quand le message a un numéro ; les enfreintes : `[r for r, respectee in regles.items() if not respectee]`
- Attendu : `(1.0, [])` pour la réponse attendue ; pour celle de Jordan, `(0.25, ['vouvoiement', 'salutation', 'longueur', 'signature', 'sans_promesse', 'commande'])`

```python
def note_charte(message: str, reponse: str) -> tuple[float, list[str]]:
  reponse = reponse.strip()
  anglais = en_anglais(message)
  salut = SALUTATION.fullmatch(reponse.split("\n")[0])
  nom = nom_signe(message)
  regles = dict(
    vouvoiement=...,
    salutation=salut is not None
    and (salut.group(1) == "Hello") == anglais
    and (salut.group(2) or "").lower() == (nom or "").lower(),
    longueur=...,
    signature=reponse.endswith(SIGNATURE_EN if anglais else SIGNATURE),
    sans_promesse=not any(...),
    sans_coordonnees=...,
    langue=...,
  )
  commande = COMMANDE.search(message)
  if commande:
    regles["commande"] = ...
  enfreintes = [r for r, respectee in regles.items() if not respectee]
  return round(1 - len(enfreintes) / len(regles), 2), enfreintes


print(note_charte(message_exemple, reponse_exemple))
print(jordan_exemple["message"], "\n\n", jordan_exemple["reponse"])
print(note_charte(jordan_exemple["message"], jordan_exemple["reponse"]))
```

### Solution

In [ ]:
def note_charte(message: str, reponse: str) -> tuple[float, list[str]]:
  reponse = reponse.strip()
  anglais = en_anglais(message)
  salut = SALUTATION.fullmatch(reponse.split("\n")[0])
  nom = nom_signe(message)
  regles = dict(
    vouvoiement=not TUTOIEMENT.search(reponse),
    salutation=salut is not None
    and (salut.group(1) == "Hello") == anglais
    and (salut.group(2) or "").lower() == (nom or "").lower(),
    longueur=40 <= len(reponse.split()) <= 120,
    signature=reponse.endswith(SIGNATURE_EN if anglais else SIGNATURE),
    sans_promesse=not any(
      PROMESSE.search(p) and not CONDITION.search(p) for p in phrases(reponse)
    ),
    sans_coordonnees=not any(
      DEMANDE.search(p) and not NEGATION.search(p) for p in phrases(reponse)
    ),
    langue=en_anglais(reponse) == anglais,
  )
  commande = COMMANDE.search(message)
  if commande:
    regles["commande"] = commande.group() in reponse
  enfreintes = [r for r, respectee in regles.items() if not respectee]
  return round(1 - len(enfreintes) / len(regles), 2), enfreintes


print(note_charte(message_exemple, reponse_exemple))
print(jordan_exemple["message"], "\n\n", jordan_exemple["reponse"])
print(note_charte(jordan_exemple["message"], jordan_exemple["reponse"]))

# La note ne voit que la forme. Un délai faux, une réponse à côté de la question
# passent toutes ses règles, et l'impératif à la deuxième personne (« Regarde sur le
# site ») échappe à TUTOIEMENT : les règles disent ce qu'on sait vérifier, le juge de
# l'exercice 4 lit le reste.

## 2. Les candidats et les paires

DPO apprend de paires : pour un même prompt, une réponse préférée et une réponse rejetée. Les plus utiles viennent du modèle lui-même (*on-policy*) : on échantillonne plusieurs réponses du modèle SFT au même message, on les note, et la meilleure et la pire forment une paire quand leur écart de note atteint `SEUIL`. Les égalités, fréquentes quand le modèle tient déjà la charte, n'apprennent rien : on les écarte.

Les messages, dans `messages` (cellule ci-dessous), viennent du jeu d'entraînement du SFT, jamais du test, avec leur catégorie et leur priorité. Un modèle SFT qui enfreint rarement la charte donne peu de paires : une seconde source les complète, des tickets de Jordan (hors test et sans bruit), dont la réponse devient la rejetée, face à la meilleure réponse du modèle. Ces paires-là sont *off-policy* : la rejetée n'est pas une réponse que le modèle écrirait.

Écrivez `echantillonne(textes, n)`, qui renvoie, pour chaque message de `textes`, `n` réponses de `modele` échantillonnées à la température `TEMPERATURE`, avec la consigne courte, par lots de quelques messages. Échantillonnez `N_CANDIDATS` réponses par message, puis construisez `paires`, un tableau d'une ligne par paire : `message`, `source` (`sft` ou `jordan`), `preferee` et `rejetee` (les sorties JSON entières), leurs notes `note_preferee` et `note_rejetee`, et `ecart`, leur différence.

- Une sortie n'est candidate que si c'est un JSON valide avec les trois clés, et, pour un message du SFT, avec sa catégorie et sa priorité : une paire ne doit pas préférer un mauvais tri.
- Pour un ticket de Jordan, la rejetée est la meilleure sortie du modèle, sa réponse remplacée par celle de Jordan : les deux ne diffèrent que par la réponse.

Combien de paires chaque source donne-t-elle ? Qu'est-ce qui distingue les paires de Jordan de celles du modèle ?

**Documentation utile**

- [`GenerationConfig`](https://huggingface.co/docs/transformers/main_classes/text_generation#transformers.GenerationConfig)
- [Les jeux de préférences de `trl`](https://huggingface.co/docs/trl/dataset_formats#preference)

In [ ]:
# Une paire : au moins une règle de la charte d'écart entre préférée et rejetée.
SEUIL = 0.1
TEMPERATURE = 1.0
N_CANDIDATS = 4
CLES = {"categorie", "priorite", "reponse"}

attendus = [lit_json(c[0]["content"]) for c in entrainement_sft["completion"]]
messages_sft = pd.DataFrame(
  dict(
    message=[p[-1]["content"] for p in entrainement_sft["prompt"]],
    categorie=[a["categorie"] for a in attendus],
    priorite=[a["priorite"] for a in attendus],
    source="sft",
  )
).sample(N_SFT, random_state=0)
# Les tickets de Jordan hors test, sans HTML, mail cité, signature de téléphone ni
# coordonnées : des messages comme ceux, nettoyés, du jeu du SFT.
BRUIT = re.compile(r"<|&\w+;|^>|Envoyé d|@|\d\d \d\d \d\d", re.MULTILINE)
jordan = tickets[
  (tickets["agent"] == "Jordan")
  & ~tickets.index.isin(test["ticket_id"])
  & ~tickets["message"].str.contains(BRUIT)
].sample(N_JORDAN, random_state=0)
messages_jordan = pd.DataFrame(
  dict(message=jordan["message"], jordan=jordan["reponse"], source="jordan")
)
messages = pd.concat([messages_sft, messages_jordan], ignore_index=True)
print(messages["source"].value_counts().to_dict())
modele = charge_modele()

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Les prompts, comme dans `genere_tout` : `tokenizer.apply_chat_template(conversation_seule(texte), tokenize=False, add_generation_prompt=True, enable_thinking=False)`
- `modele.generate(**entrees, max_new_tokens=384, do_sample=True, temperature=TEMPERATURE, num_return_sequences=n)` : `n` réponses par prompt, à la suite (les `n` premières pour le premier message, et ainsi de suite) ; `top_k` et `top_p` restent ceux de la configuration de Qwen3 (20 et 0,95)
- `torch.manual_seed(0)` avant de tirer : le même tirage à chaque exécution
- Une sortie candidate : `CLES <= objet.keys()`, avec `objet = lit_json(sortie)`, et pour un message `sft`, `(objet["categorie"], objet["priorite"]) == (ligne.categorie, ligne.priorite)`
- Les candidates d'un message triées par note, `notees.sort(key=lambda x: x[0])` sur une liste de `(note, sortie)` : la pire en premier, la meilleure en dernier
- La rejetée d'un ticket de Jordan, au format des sorties du modèle : `json.dumps(dict(lit_json(preferee), reponse=ligne.jordan), ensure_ascii=False)`
- Sur GPU, comptez un quart d'heure pour les 560 réponses ; sans GPU, une dizaine de minutes pour 20

```python
torch.manual_seed(0)


def echantillonne(textes: list[str], n: int, taille_lot: int = 4) -> list[list[str]]:
  candidats = []
  for debut in range(0, len(textes), taille_lot):
    prompts = [...]
    entrees = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
      sorties = modele.generate(...)
    lot = tokenizer.batch_decode(
      sorties[:, entrees.input_ids.shape[1] :], skip_special_tokens=True
    )
    candidats += [lot[i : i + n] for i in range(0, len(lot), n)]
  return candidats


def note_sortie(ligne, sortie: str) -> float | None:
  """La note d'une sortie, ou None si elle n'est pas candidate."""
  objet = lit_json(sortie)
  if ...:
    return None
  if ligne.source == "sft" and ...:
    return None
  return note_charte(ligne.message, str(objet["reponse"]))[0]


debut = time.time()
candidats = echantillonne(list(messages["message"]), N_CANDIDATS)
print(
  f"{(time.time() - debut) / 60:.1f} min pour {len(messages) * N_CANDIDATS} réponses"
)
lignes = []
for ligne, sorties in zip(messages.itertuples(), candidats):
  notees = []
  for sortie in sorties:
    note = note_sortie(ligne, sortie)
    if note is not None:
      notees.append((note, sortie.strip()))
  if not notees:
    continue
  notees.sort(key=lambda x: x[0])
  (note_rejetee, rejetee), (note_preferee, preferee) = notees[0], notees[-1]
  if ligne.source == "jordan":
    rejetee = ...
    note_rejetee = ...
  if note_preferee - note_rejetee >= SEUIL:
    lignes.append(
      dict(
        message=ligne.message,
        source=ligne.source,
        preferee=preferee,
        rejetee=rejetee,
        note_preferee=note_preferee,
        note_rejetee=note_rejetee,
        ecart=round(note_preferee - note_rejetee, 2),
      )
    )
paires = pd.DataFrame(lignes)
print(paires["source"].value_counts().to_dict())
paires.head()
```

### Solution

In [ ]:
torch.manual_seed(0)


def echantillonne(textes: list[str], n: int, taille_lot: int = 4) -> list[list[str]]:
  candidats = []
  for debut in range(0, len(textes), taille_lot):
    prompts = [
      tokenizer.apply_chat_template(
        conversation_seule(texte),
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
      )
      for texte in textes[debut : debut + taille_lot]
    ]
    entrees = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
      sorties = modele.generate(
        **entrees,
        max_new_tokens=384,
        do_sample=True,
        temperature=TEMPERATURE,
        num_return_sequences=n,
      )
    lot = tokenizer.batch_decode(
      sorties[:, entrees.input_ids.shape[1] :], skip_special_tokens=True
    )
    candidats += [lot[i : i + n] for i in range(0, len(lot), n)]
  return candidats


def note_sortie(ligne, sortie: str) -> float | None:
  """La note d'une sortie, ou None si elle n'est pas candidate."""
  objet = lit_json(sortie)
  if not CLES <= objet.keys():
    return None
  if ligne.source == "sft" and (objet["categorie"], objet["priorite"]) != (
    ligne.categorie,
    ligne.priorite,
  ):
    return None
  return note_charte(ligne.message, str(objet["reponse"]))[0]


debut = time.time()
candidats = echantillonne(list(messages["message"]), N_CANDIDATS)
print(
  f"{(time.time() - debut) / 60:.1f} min pour {len(messages) * N_CANDIDATS} réponses"
)
lignes = []
for ligne, sorties in zip(messages.itertuples(), candidats):
  notees = []
  for sortie in sorties:
    note = note_sortie(ligne, sortie)
    if note is not None:
      notees.append((note, sortie.strip()))
  if not notees:
    continue
  notees.sort(key=lambda x: x[0])
  (note_rejetee, rejetee), (note_preferee, preferee) = notees[0], notees[-1]
  if ligne.source == "jordan":
    rejetee = json.dumps(
      dict(lit_json(preferee), reponse=ligne.jordan), ensure_ascii=False
    )
    note_rejetee = note_charte(ligne.message, ligne.jordan)[0]
  if note_preferee - note_rejetee >= SEUIL:
    lignes.append(
      dict(
        message=ligne.message,
        source=ligne.source,
        preferee=preferee,
        rejetee=rejetee,
        note_preferee=note_preferee,
        note_rejetee=note_rejetee,
        ecart=round(note_preferee - note_rejetee, 2),
      )
    )
paires = pd.DataFrame(lignes)
print(paires["source"].value_counts().to_dict())
paires.head()

# Les paires sft opposent deux réponses que le modèle écrit vraiment : DPO déplace la
# probabilité entre elles, là où le modèle hésite. Celles de Jordan opposent une
# réponse du modèle à un texte qu'il n'écrirait pas (tutoiement, une phrase) : la
# rejetée est déjà très improbable, la paire est facile, et elle apprend surtout à
# ne pas faire ce que le modèle ne fait déjà presque plus. Elles complètent les
# paires du modèle, elles ne les remplacent pas.

## 3. Contrôler le jeu de préférences

Un jeu de préférences se contrôle avant d'entraîner, comme un jeu d'entraînement : un biais qu'il porte, DPO l'apprend. Sur `paires`, regardez :

- le nombre de paires par source, et la distribution de leurs écarts de note ;
- les règles que les rejetées enfreignent : ce que les paires apprennent vraiment ;
- le biais de longueur : dans quelle part des paires, par source, la réponse préférée est-elle la plus longue ? Si c'est presque toujours elle, DPO peut apprendre « plus long, c'est mieux » ;
- les doublons : des paires identiques, et des rejetées qui se répètent (Jordan a ses formules) ; retirez les paires en double.

Puis mettez les paires au format conversationnel de `trl`, dans un `Dataset` aux colonnes `prompt` (la conversation de la consigne courte et du message, `conversation_seule`), `chosen` et `rejected` (chacune une liste d'un message `assistant`), et coupez-le en entraînement et validation, 10 % pour la validation, dans `decoupe`.

Quel biais ce jeu porte-t-il ? Que changeriez-vous dans sa construction ?

**Documentation utile**

- [`collections.Counter`](https://docs.python.org/3/library/collections.html#collections.Counter)
- [`DataFrame.duplicated`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html)
- [`Dataset.from_list`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.from_list)
- [`Dataset.train_test_split`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.train_test_split)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- La réponse d'une sortie : `reponse_de(sortie)`, à écrire, `str(lit_json(sortie).get("reponse", ""))`
- `paires.groupby("source")["ecart"].describe()` : la distribution des écarts par source
- `collections.Counter(...)` sur les règles de `note_charte(ligne.message, reponse_de(ligne.rejetee))[1]`, pour chaque paire ; `.most_common()` les range
- Les longueurs : `paires["preferee"].map(lambda s: len(reponse_de(s).split()))` ; `(mots_preferee > mots_rejetee).groupby(paires["source"]).mean()`, la part par source
- `paires.duplicated(["message", "preferee", "rejetee"])` : vrai pour chaque paire déjà vue ; `.groupby(paires["source"]).nunique()` sur les rejetées, leurs textes distincts
- Un exemple de `trl` : `dict(prompt=conversation_seule(message), chosen=[dict(role="assistant", content=preferee)], rejected=[...])` ; `Dataset.from_list(exemples)`
- `jeu.train_test_split(test_size=0.1, seed=0)` renvoie un `DatasetDict`, `decoupe["train"]` et `decoupe["test"]`

```python
def reponse_de(sortie: str) -> str:
  return str(lit_json(sortie).get("reponse", ""))


print(paires.groupby("source")["ecart"].describe().round(2))
enfreintes = collections.Counter(
  regle for ligne in paires.itertuples() for regle in note_charte(...)[1]
)
print("Règles enfreintes par les rejetées :", dict(enfreintes.most_common()))
mots_preferee = paires["preferee"].map(lambda s: len(reponse_de(s).split()))
mots_rejetee = ...
plus_longue = (mots_preferee > mots_rejetee).groupby(paires["source"]).mean()
print("La préférée est la plus longue :", plus_longue.round(2).to_dict())
print("Mots, préférées et rejetées :", round(mots_preferee.mean()), round(...))
doublons = paires.duplicated(...)
print("Paires en double :", doublons.sum())
rejetees = paires["rejetee"].map(reponse_de)
print("Rejetées distinctes :", rejetees.groupby(paires["source"]).nunique().to_dict())
paires = paires[~doublons].reset_index(drop=True)

jeu = Dataset.from_list(
  [
    dict(
      prompt=...,
      chosen=[dict(role="assistant", content=ligne.preferee)],
      rejected=...,
    )
    for ligne in paires.itertuples()
  ]
)
decoupe = jeu.train_test_split(...)
print(decoupe)
```

### Solution

In [ ]:
def reponse_de(sortie: str) -> str:
  return str(lit_json(sortie).get("reponse", ""))


print(paires.groupby("source")["ecart"].describe().round(2))
enfreintes = collections.Counter(
  regle
  for ligne in paires.itertuples()
  for regle in note_charte(ligne.message, reponse_de(ligne.rejetee))[1]
)
print("Règles enfreintes par les rejetées :", dict(enfreintes.most_common()))
mots_preferee = paires["preferee"].map(lambda s: len(reponse_de(s).split()))
mots_rejetee = paires["rejetee"].map(lambda s: len(reponse_de(s).split()))
plus_longue = (mots_preferee > mots_rejetee).groupby(paires["source"]).mean()
print("La préférée est la plus longue :", plus_longue.round(2).to_dict())
print(
  "Mots, préférées et rejetées :",
  round(mots_preferee.mean()),
  round(mots_rejetee.mean()),
)
doublons = paires.duplicated(["message", "preferee", "rejetee"])
print("Paires en double :", doublons.sum())
rejetees = paires["rejetee"].map(reponse_de)
print("Rejetées distinctes :", rejetees.groupby(paires["source"]).nunique().to_dict())
paires = paires[~doublons].reset_index(drop=True)

jeu = Dataset.from_list(
  [
    dict(
      prompt=conversation_seule(ligne.message),
      chosen=[dict(role="assistant", content=ligne.preferee)],
      rejected=[dict(role="assistant", content=ligne.rejetee)],
    )
    for ligne in paires.itertuples()
  ]
)
decoupe = jeu.train_test_split(test_size=0.1, seed=0)
print(decoupe)

# Les paires de Jordan préfèrent toujours la plus longue (ses réponses font une ou
# deux phrases) et leurs rejetées se répètent : elles poussent vers « plus long,
# mieux », que l'exercice 6 surveillera (`mots`). Les paires du modèle portent ses
# vraies hésitations. Remèdes : plafonner la part de Jordan, garder les paires du
# modèle où la préférée est la plus courte, échantillonner plus de réponses par
# message pour en avoir davantage.

## 4. Un juge en contrôle

Les notes viennent de règles : un jugement sur l'ensemble de la réponse leur donne-t-il raison ? Sur un vrai projet, des annotateurs comparent un échantillon de paires sans voir la préférence calculée. Ici, le juge de l'atelier tient ce rôle, dans les deux ordres, avec `GRILLE_PREFERENCE` (cellule ci-dessous) : la charte seule, sans réponse de référence, que les tickets de Jordan n'ont pas.

Sur `N_CONTROLE` paires tirées au hasard, dans `controle`, faites comparer au juge la réponse préférée et la rejetée avec `juge_deux_ordres`, et rangez chaque verdict dans une colonne `juge` : `accord` (le juge choisit la préférée dans les deux ordres), `desaccord` (la rejetée dans les deux), `indecis` (son verdict suit la position). Comptez-les par source, et lisez les paires en désaccord.

Le juge confirme-t-il la préférence calculée ? Que faire d'une paire en désaccord : la règle a-t-elle tort, ou le juge ?

**Documentation utile**

- [`pd.crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)

In [ ]:
del modele
libere()
juge = charge_juge()
GRILLE_PREFERENCE = f"""Tu compares deux réponses du service client de Biclou & Cie,
magasins de vélos, au même message d'un client. On te donne le message, puis les
réponses A et B. Dis laquelle suit le mieux la charte ci-dessous, sur la forme et le
ton : le vouvoiement, la salutation et la signature de la charte, 40 à 120 mots, le
numéro de commande rappelé, aucune promesse de remboursement avant contrôle, ni
coordonnées bancaires ni mot de passe demandés, la langue du client.
Réponds uniquement par un objet JSON : {{"meilleure": "A"}} ou {{"meilleure": "B"}}.

{charte}"""


def demande_preference(message: str, reponse_a: str, reponse_b: str) -> str:
  return (
    f"Message du client :\n{message}\n\n"
    f"Réponse A :\n{reponse_a}\n\n"
    f"Réponse B :\n{reponse_b}"
  )

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `paires.sample(min(N_CONTROLE, len(paires)), random_state=0)` : l'échantillon
- Pour chaque paire, `demande_preference(ligne.message, preferee, rejetee)` dans `demandes`, et l'inverse, `demande_preference(ligne.message, rejetee, preferee)`, dans `inversees`, avec `reponse_de` pour les réponses
- `juge_deux_ordres(GRILLE_PREFERENCE, demandes, inversees)` renvoie `"premiere"` (la préférée gagne dans les deux ordres), `"seconde"` ou `None` ; un dictionnaire les traduit en `accord`, `desaccord` et `indecis`
- `pd.crosstab(controle["source"], controle["juge"])` : les verdicts par source
- Sur GPU, deux à trois minutes pour 60 verdicts ; sans GPU, une demi-minute environ par verdict

```python
controle = paires.sample(...)
demandes, inversees = [], []
for ligne in controle.itertuples():
  preferee, rejetee = reponse_de(ligne.preferee), reponse_de(ligne.rejetee)
  demandes.append(...)
  inversees.append(...)
gagnants = juge_deux_ordres(...)
verdicts = {"premiere": "accord", "seconde": "desaccord", None: "indecis"}
controle["juge"] = [verdicts[g] for g in gagnants]
print(pd.crosstab(...))
print(f"Accord : {(controle['juge'] == 'accord').mean():.0%}")
for ligne in controle[controle["juge"] == "desaccord"].head(3).itertuples():
  print(f"--- {ligne.source}, écart {ligne.ecart}\n{ligne.message}\n")
  print("Préférée :", reponse_de(ligne.preferee))
  print("Rejetée :", reponse_de(ligne.rejetee), "\n")
```

### Solution

In [ ]:
controle = paires.sample(min(N_CONTROLE, len(paires)), random_state=0)
demandes, inversees = [], []
for ligne in controle.itertuples():
  preferee, rejetee = reponse_de(ligne.preferee), reponse_de(ligne.rejetee)
  demandes.append(demande_preference(ligne.message, preferee, rejetee))
  inversees.append(demande_preference(ligne.message, rejetee, preferee))
gagnants = juge_deux_ordres(GRILLE_PREFERENCE, demandes, inversees)
verdicts = {"premiere": "accord", "seconde": "desaccord", None: "indecis"}
controle["juge"] = [verdicts[g] for g in gagnants]
print(pd.crosstab(controle["source"], controle["juge"]))
print(f"Accord : {(controle['juge'] == 'accord').mean():.0%}")
for ligne in controle[controle["juge"] == "desaccord"].head(3).itertuples():
  print(f"--- {ligne.source}, écart {ligne.ecart}\n{ligne.message}\n")
  print("Préférée :", reponse_de(ligne.preferee))
  print("Rejetée :", reponse_de(ligne.rejetee), "\n")

# Une paire en désaccord se lit : la préférée peut gagner par une règle de forme
# (la longueur, le numéro de commande) et dire faux, ou plus sèchement, ce que les
# règles ne voient pas ; ou le juge se trompe. Une paire où il a raison sort du jeu ;
# s'il y en a beaucoup, c'est la note qu'on revoit. Les indécis disent ce que vaut le
# juge sur ces paires. Sur un vrai projet, ce contrôle est humain, sur 50 à 100
# paires, et l'accord entre deux annotateurs se mesure aussi.

## 5. L'alignement DPO avec `trl`

DPO entraîne directement sur les paires, sans modèle de récompense : il monte la probabilité de la préférée et baisse celle de la rejetée, par rapport à un modèle de référence gelé, ici le modèle SFT, dont `beta` règle combien il peut s'écarter. On entraîne l'adaptateur SFT lui-même : chargé entraînable et passé à `DPOTrainer` sans `peft_config`, il est copié par `trl` dans un second adaptateur gelé, `ref`, qui sert de référence.

Le juge est libéré (cellule ci-dessous). Chargez `modele`, le modèle de base de `charge_base()` avec l'adaptateur `ADAPTATEUR` entraînable, et préparez `trainer`, un `DPOTrainer` sur `decoupe` : `beta` de 0,1, un taux d'apprentissage de 2e-5, dix fois plus bas que celui du SFT, `EPOQUES` époques, `LOT` paires par pas et `ACCUMULATION` pas par mise à jour, la précision et l'optimiseur du TP d'entraînement, un journal à chaque pas et une évaluation à chaque époque. Vérifiez les adaptateurs du modèle (`modele.peft_config`), entraînez, puis lisez dans `trainer.state.log_history` la perte et les récompenses implicites : `rewards/chosen`, `rewards/rejected`, `rewards/margins`, `rewards/accuracies`.

Enregistrez enfin l'adaptateur entraîné, `default`, seul, dans `DOSSIER_DPO`, et, si vous avez le jeton `HF_TOKEN`, publiez-le en **privé** sous `<nom>/support-dpo-lora`, comme au TP d'entraînement.

Que disent les récompenses : la préférée monte-t-elle, ou la rejetée baisse-t-elle surtout ? Que mesure `rewards/accuracies` sur le jeu de validation ?

**Documentation utile**

- [`DPOConfig`](https://huggingface.co/docs/trl/dpo_trainer#trl.DPOConfig)
- [`DPOTrainer`](https://huggingface.co/docs/trl/dpo_trainer#trl.DPOTrainer)
- [Les métriques de `DPOTrainer`](https://huggingface.co/docs/trl/dpo_trainer#logged-metrics)
- [`PeftModel.save_pretrained`](https://huggingface.co/docs/peft/package_reference/peft_model#peft.PeftModel.save_pretrained)
- [`HfApi.upload_folder`](https://huggingface.co/docs/huggingface_hub/package_reference/hf_api#huggingface_hub.HfApi.upload_folder)

In [ ]:
del juge
libere()
DOSSIER_DPO = "support-dpo-lora"
# Paires par pas et pas accumulés par mise à jour : 8 paires sur GPU, 2 sans.
LOT = 4 if GPU else 2
ACCUMULATION = 2 if GPU else 1
EPOQUES = 2 if GPU else 1

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `PeftModel.from_pretrained(charge_base(), ADAPTATEUR, is_trainable=True)` : sans `is_trainable`, l'adaptateur est chargé gelé, pour l'inférence
- `DPOConfig(output_dir="dpo", beta=0.1, learning_rate=2e-5, num_train_epochs=EPOQUES, per_device_train_batch_size=LOT, per_device_eval_batch_size=LOT, gradient_accumulation_steps=ACCUMULATION, ...)`
- Du TP d'entraînement : `lr_scheduler_type="cosine"`, `warmup_steps=0.1`, `optim="paged_adamw_8bit" if GPU else "adamw_torch"`, `gradient_checkpointing=GPU`, `bf16=BF16`, `fp16=GPU and not BF16`, `max_length=1024` ; puis `logging_steps=1`, `eval_strategy="epoch"`, `save_strategy="no"` (pas de checkpoint), `report_to="none"`
- `DPOTrainer(model=modele, args=config, train_dataset=decoupe["train"], eval_dataset=decoupe["test"], processing_class=tokenizer)` : sans `ref_model` ni `peft_config`, `list(modele.peft_config)` vaut ensuite `['default', 'ref']`
- `pd.DataFrame(trainer.state.log_history)` : une ligne par journal ; celles de l'entraînement ont une `loss`, celles de l'évaluation une `eval_loss` et les mêmes métriques préfixées par `eval_` ; `logps/chosen` et `logps/rejected` donnent les log-probabilités elles-mêmes
- `trainer.model.save_pretrained(DOSSIER_DPO, selected_adapters=["default"])` : sans `selected_adapters`, l'adaptateur `ref` serait enregistré aussi
- Le jeton et la publication, comme au TP d'entraînement : `HfApi(token=jeton)`, `api.create_repo(depot, private=True, exist_ok=True)`, `api.upload_folder(folder_path=DOSSIER_DPO, repo_id=depot)`
- Sur GPU, une dizaine de minutes ; sans GPU, quelques minutes par pas

```python
modele = PeftModel.from_pretrained(...)
config = DPOConfig(
  output_dir="dpo",
  beta=0.1,
  learning_rate=2e-5,
  num_train_epochs=EPOQUES,
  per_device_train_batch_size=LOT,
  per_device_eval_batch_size=LOT,
  gradient_accumulation_steps=ACCUMULATION,
  lr_scheduler_type="cosine",
  warmup_steps=0.1,
  optim=...,
  gradient_checkpointing=GPU,
  bf16=BF16,
  fp16=...,
  max_length=1024,
  logging_steps=1,
  eval_strategy=...,
  save_strategy="no",
  report_to="none",
)
trainer = DPOTrainer(...)
print("Adaptateurs :", list(modele.peft_config))
debut = time.time()
trainer.train()
print(f"{(time.time() - debut) / 60:.1f} min")
METRIQUES = [
  "loss",
  "rewards/chosen",
  "rewards/rejected",
  "rewards/margins",
  "rewards/accuracies",
]
historique = pd.DataFrame(trainer.state.log_history)
print(historique.dropna(subset=["loss"]).set_index("step")[METRIQUES].round(3))
evaluations = historique.dropna(subset=["eval_loss"]).set_index("epoch")
print(evaluations[["eval_" + m for m in METRIQUES]].round(3))

trainer.model.save_pretrained(...)
print(os.listdir(DOSSIER_DPO))
jeton = os.environ.get("HF_TOKEN")
try:
  from google.colab import userdata

  jeton = userdata.get("HF_TOKEN")
except Exception:
  pass
if jeton:
  api = HfApi(token=jeton)
  depot_dpo = f"{api.whoami()['name']}/support-dpo-lora"
  api.create_repo(...)
  api.upload_folder(...)
  print("Publié :", depot_dpo)
else:
  print("Pas de jeton HF_TOKEN : l'adaptateur reste dans", DOSSIER_DPO)
```

### Solution

In [ ]:
modele = PeftModel.from_pretrained(charge_base(), ADAPTATEUR, is_trainable=True)
config = DPOConfig(
  output_dir="dpo",
  beta=0.1,
  learning_rate=2e-5,
  num_train_epochs=EPOQUES,
  per_device_train_batch_size=LOT,
  per_device_eval_batch_size=LOT,
  gradient_accumulation_steps=ACCUMULATION,
  lr_scheduler_type="cosine",
  warmup_steps=0.1,
  optim="paged_adamw_8bit" if GPU else "adamw_torch",
  gradient_checkpointing=GPU,
  bf16=BF16,
  fp16=GPU and not BF16,
  max_length=1024,
  logging_steps=1,
  eval_strategy="epoch",
  save_strategy="no",
  report_to="none",
)
trainer = DPOTrainer(
  model=modele,
  args=config,
  train_dataset=decoupe["train"],
  eval_dataset=decoupe["test"],
  processing_class=tokenizer,
)
print("Adaptateurs :", list(modele.peft_config))
debut = time.time()
trainer.train()
print(f"{(time.time() - debut) / 60:.1f} min")
METRIQUES = [
  "loss",
  "rewards/chosen",
  "rewards/rejected",
  "rewards/margins",
  "rewards/accuracies",
]
historique = pd.DataFrame(trainer.state.log_history)
print(historique.dropna(subset=["loss"]).set_index("step")[METRIQUES].round(3))
evaluations = historique.dropna(subset=["eval_loss"]).set_index("epoch")
print(evaluations[["eval_" + m for m in METRIQUES]].round(3))

trainer.model.save_pretrained(DOSSIER_DPO, selected_adapters=["default"])
print(os.listdir(DOSSIER_DPO))
jeton = os.environ.get("HF_TOKEN")
try:
  from google.colab import userdata

  jeton = userdata.get("HF_TOKEN")
except Exception:
  pass
if jeton:
  api = HfApi(token=jeton)
  depot_dpo = f"{api.whoami()['name']}/support-dpo-lora"
  api.create_repo(depot_dpo, private=True, exist_ok=True)
  api.upload_folder(folder_path=DOSSIER_DPO, repo_id=depot_dpo)
  print("Publié :", depot_dpo)
else:
  print("Pas de jeton HF_TOKEN : l'adaptateur reste dans", DOSSIER_DPO)

# rewards/chosen et rewards/rejected : beta fois l'écart de log-probabilité de la
# préférée, et de la rejetée, entre le modèle et la référence ; margins, leur
# différence ; accuracies, la part des paires où la préférée l'emporte. Au premier
# pas, le modèle est la référence : tout vaut 0 et la perte log(2), 0,693. Souvent,
# la rejetée baisse plus que la préférée ne monte : DPO creuse l'écart sans
# forcément mieux écrire. Sur la validation, accuracies dit si la préférence
# s'étend à des paires que le modèle n'a pas vues.

## 6. Avant / après : les mesures

Le modèle de l'entraînement est libéré et rechargé pour l'évaluation (cellule ci-dessous) : le modèle SFT de `charge_modele()`, et à côté, sous le nom `dpo`, l'adaptateur aligné. `modele.set_adapter("default")` en fait le modèle SFT, `modele.set_adapter("dpo")` le modèle aligné : `ADAPTATEURS` les nomme.

Avec le protocole de l'atelier, mesurez les deux sur `test_part` avec la consigne courte : `evalue` (le JSON, le tri, `respecte_charte`, la longueur en `mots`), la note moyenne de `note_charte` et les règles enfreintes. Posez-leur aussi les questions générales de l'atelier, `QUESTIONS`, avec le message `system` `ASSISTANT` : la part de bonnes réponses, et la part de réponses qui sont un ticket en JSON (la fuite du format). Rassemblez tout dans un tableau `avant_apres`, une ligne par modèle, et gardez leurs sorties sur le test dans `sorties_test`, pour l'exercice 7.

Qu'est-ce qui a bougé ? La longueur des réponses a-t-elle suivi le biais du jeu ? Le tri, le format et les questions générales ont-ils tenu ?

**Documentation utile**

- [`PeftModel.set_adapter`](https://huggingface.co/docs/peft/package_reference/peft_model#peft.PeftModel.set_adapter)

In [ ]:
del trainer, modele
libere()
modele = charge_modele()
modele.load_adapter(DOSSIER_DPO, adapter_name="dpo")
ADAPTATEURS = {"SFT": "default", "DPO": "dpo"}

# Les questions générales de l'atelier, et un mot que chaque bonne réponse contient.
ASSISTANT = "Tu es un assistant généraliste. Réponds en une ou deux phrases."
QUESTIONS = {
  "Quelle est la capitale de l'Australie ?": "Canberra",
  "Combien font 17 fois 3 ?": "51",
  "Qui a écrit Les Misérables ?": "Hugo",
  "Combien de jours compte une année bissextile ?": "366",
  "Traduis en anglais : « Le chat dort sur le canapé. »": "cat",
  "Quel gaz les plantes absorbent-elles pour la photosynthèse ?": "carbone",
}

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `modele.set_adapter(adaptateur)`, puis `genere_tout(modele, conversations)` : les sorties du modèle de cet adaptateur
- `evalue(sorties)` renvoie une `pd.Series` : ajoutez-lui `note_charte`, la moyenne des notes, et les parts des questions générales (`mesures[nom]["note_charte"] = ...`)
- Une question générale : `[{"role": "system", "content": ASSISTANT}, {"role": "user", "content": question}]`, et `genere_tout(modele, generales, max_tokens=128)` ; une bonne réponse, `attendu.lower() in reponse.lower()` ; un ticket, `"categorie" in lit_json(reponse)`
- Les règles enfreintes de chaque modèle : `collections.Counter` sur les listes de `note_charte`
- Sur GPU, cinq minutes environ par modèle

```python
conversations = [conversation_seule(m) for m in test_part["message"]]
generales = [
  [{"role": "system", "content": ASSISTANT}, {"role": "user", "content": q}]
  for q in QUESTIONS
]
sorties_test, mesures = {}, {}
for nom, adaptateur in ADAPTATEURS.items():
  modele.set_adapter(...)
  sorties_test[nom] = genere_tout(modele, conversations)
  notes = [
    note_charte(m, reponse_de(s))
    for m, s in zip(test_part["message"], sorties_test[nom])
  ]
  hors_sujet = genere_tout(...)
  mesures[nom] = evalue(sorties_test[nom])
  mesures[nom]["note_charte"] = round(sum(n for n, _ in notes) / len(notes), 2)
  bonnes = sum(a.lower() in r.lower() for r, a in zip(hors_sujet, QUESTIONS.values()))
  mesures[nom]["bonnes_generales"] = round(bonnes / len(QUESTIONS), 2)
  mesures[nom]["tickets_json"] = round(... / len(QUESTIONS), 2)
  print(nom, "enfreint :", dict(collections.Counter(...)))
avant_apres = pd.DataFrame(mesures).T
print(sorties_test["DPO"][0])
avant_apres
```

### Solution

In [ ]:
conversations = [conversation_seule(m) for m in test_part["message"]]
generales = [
  [{"role": "system", "content": ASSISTANT}, {"role": "user", "content": q}]
  for q in QUESTIONS
]
sorties_test, mesures = {}, {}
for nom, adaptateur in ADAPTATEURS.items():
  modele.set_adapter(adaptateur)
  sorties_test[nom] = genere_tout(modele, conversations)
  notes = [
    note_charte(m, reponse_de(s))
    for m, s in zip(test_part["message"], sorties_test[nom])
  ]
  hors_sujet = genere_tout(modele, generales, max_tokens=128)
  mesures[nom] = evalue(sorties_test[nom])
  mesures[nom]["note_charte"] = round(sum(n for n, _ in notes) / len(notes), 2)
  bonnes = sum(a.lower() in r.lower() for r, a in zip(hors_sujet, QUESTIONS.values()))
  mesures[nom]["bonnes_generales"] = round(bonnes / len(QUESTIONS), 2)
  tickets_json = sum("categorie" in lit_json(r) for r in hors_sujet)
  mesures[nom]["tickets_json"] = round(tickets_json / len(QUESTIONS), 2)
  print(nom, "enfreint :", dict(collections.Counter(r for _, e in notes for r in e)))
avant_apres = pd.DataFrame(mesures).T
print(sorties_test["DPO"][0])
avant_apres

# Un alignement sur la forme se lit d'abord dans note_charte et les règles
# enfreintes. Les colonnes de l'atelier disent ce qu'il a pu coûter : un tri qui
# baisse, un JSON moins souvent valide, des réponses plus longues (le biais de
# longueur du jeu), un modèle qui répond par un ticket à une question générale.

## 7. Avant / après : le juge par paires

Le modèle est libéré et le juge rechargé (cellule ci-dessous). Comme à l'exercice 5 de l'atelier, faites comparer au juge, avec `GRILLE_PAIRE` et `demande_paire`, la réponse du modèle SFT et celle du modèle aligné à chaque ticket de `duel`, les `N_DUEL` premiers de `test_part` (50 sur GPU), dans les deux ordres (`juge_deux_ordres`). Le taux de victoire : la part des tickets que chaque modèle gagne dans les deux ordres, et celle des indécis.

Concluez : l'alignement vaut-il ce qu'il a coûté (les paires, leur contrôle, l'entraînement, une évaluation de plus) ? Qu'a-t-il changé, que n'a-t-il pas changé ?

In [ ]:
del modele
libere()
juge = charge_juge()
duel = test_part.head(N_DUEL)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Les réponses comparées, comme dans l'atelier : `str(lit_json(s).get("reponse", s))` pour chaque sortie de `sorties_test["SFT"]` et de `sorties_test["DPO"]`, sur les `len(duel)` premiers tickets
- `demande_paire(ligne, avant, apres)` dans `demandes`, `demande_paire(ligne, apres, avant)` dans `inversees`, pour chaque `ligne` de `duel.itertuples()`
- `"premiere"` : le modèle SFT gagne dans les deux ordres ; `"seconde"` : le modèle aligné
- `pd.Series(noms).value_counts(normalize=True)` : les parts
- Sur GPU, cinq minutes environ pour 100 verdicts

```python
avant = [str(lit_json(s).get("reponse", s)) for s in sorties_test["SFT"][: len(duel)]]
apres = [...]
demandes = [demande_paire(l, a, b) for l, a, b in zip(duel.itertuples(), avant, apres)]
inversees = [...]
gagnants = juge_deux_ordres(...)
noms = {"premiere": "SFT", "seconde": "DPO", None: "indécis"}
victoires = pd.Series([noms[g] for g in gagnants]).value_counts(normalize=True)
print("Taux de victoire :", victoires.round(2).to_dict())
avant_apres
```

### Solution

In [ ]:
avant = [str(lit_json(s).get("reponse", s)) for s in sorties_test["SFT"][: len(duel)]]
apres = [str(lit_json(s).get("reponse", s)) for s in sorties_test["DPO"][: len(duel)]]
demandes = [demande_paire(l, a, b) for l, a, b in zip(duel.itertuples(), avant, apres)]
inversees = [demande_paire(l, b, a) for l, a, b in zip(duel.itertuples(), avant, apres)]
gagnants = juge_deux_ordres(GRILLE_PAIRE, demandes, inversees)
noms = {"premiere": "SFT", "seconde": "DPO", None: "indécis"}
victoires = pd.Series([noms[g] for g in gagnants]).value_counts(normalize=True)
print("Taux de victoire :", victoires.round(2).to_dict())
avant_apres

# Le juge lit aussi les faits et l'étape suivante, que l'alignement n'a pas visés :
# beaucoup d'indécis, ou des victoires partagées, disent que les deux modèles se
# valent sur ce qu'il juge. L'alignement vaut son coût s'il corrige ce que voient les
# clients (le ton, une promesse, une salutation) sans rien perdre sur le tri, le
# format ni les questions générales. Si le modèle SFT tenait déjà la charte, des
# exemples corrigés dans le jeu du SFT coûtent moins cher qu'un jeu de préférences.

## À retenir

- DPO apprend de paires préférée / rejetée, contre un modèle de référence gelé, ici le modèle SFT : ni modèle de récompense ni génération pendant l'entraînement, contrairement au RLHF avec PPO. Avec LoRA, la référence n'est qu'une copie gelée de l'adaptateur.
- Un critère que des règles vérifient (la forme de la charte) donne des paires sans annotateur : on échantillonne le modèle (*on-policy*), et la meilleure et la pire réponse forment une paire quand leur écart compte. Une source *off-policy*, comme les réponses de Jordan, complète, mais apprend surtout ce que le modèle ne fait déjà plus.
- Le jeu se contrôle avant d'entraîner : les écarts, les règles en jeu, le biais de longueur, les doublons, et l'accord d'un juge ou d'annotateurs sur un échantillon.
- `rewards/margins` et `rewards/accuracies` disent si le modèle sépare les paires, pas s'il répond mieux : la comparaison avant / après avec le protocole de l'atelier, questions hors du service client comprises, le dit.